# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hammadkhaliq-del/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Contract with the w04 baseline, stated up front:** the ML-07 rule scores `decent_position` (`avg_position_month <= 20`) AND `ctr_underperforming` (CTR below 70% of the position band's average) AND weights by `impressions_month`. It has never been scored against an independent outcome -- only against its own top reason code, which the w04 notebook itself flags as circular. This notebook builds that outcome (`is_declining_next`, forward-looking, April vs March) and re-scores the *exact same rule* against it, on the *exact same held-out rows* the model is evaluated on.

**Feature policy, carried over from the w04 fix:** `dim_content` (`word_count`, `content_type`, `last_optimized_date`) is current-state, not point-in-time -- that's what broke the staleness signal in w04. The same risk applies to using those columns as model features here, so they're excluded from the default feature set below, same as they were excluded from the rule's scoring logic (kept only as non-decision context there).

In [1]:
%pip install -q duckdb scikit-learn

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('AccessToken')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEATURE_MONTH = "2026-03"   # same month the w04 baseline scored
LABEL_MONTH   = "2026-04"   # NEXT month -- used only to build the ground-truth outcome, NEVER as a feature

DAILY_FEATURE = f"{REL}/fact_content_daily_performance/month={FEATURE_MONTH}/data_0.parquet"
DAILY_LABEL   = f"{REL}/fact_content_daily_performance/month={LABEL_MONTH}/data_0.parquet"

# Hard check, not an assumption left unverified. NOTE: if this throws a file-not-found / HTTP error rather than
# failing the assert below, the April partition may not use the same single-file 'data_0.parquet' naming w04 relied
# on for March -- swap in '{REL}/fact_content_daily_performance/month={LABEL_MONTH}/*.parquet' (wildcard) and retry.
probe = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{DAILY_LABEL}')").df()
print(f'{LABEL_MONTH} partition row count:', probe['n'].iloc[0])
assert probe['n'].iloc[0] > 0, (
    f'{LABEL_MONTH} has no rows in this snapshot under this path. Stop and either fix the path (see comment above) '
    'or pick a different label design rather than pushing on with an empty label source.'
)
print('Ready.')

2026-04 partition row count: 10424730
Ready.


## 1. Method choice and why

**The label, built first because it decides everything else:** `is_declining_next = 1` when a page's GSC impressions drop more than 20% from `FEATURE_MONTH` to `LABEL_MONTH` (same >20% threshold the starter kit's own `trend_direction` label uses), restricted to pages with `gsc_data_available IS TRUE` in *both* months and non-zero March impressions.

**Method choice — provisional until the class-balance check below runs:**
- **Logistic Regression** — the honest baseline-*model*. If it can't beat the hand-rule, that's a real, reportable result.
- **Random Forest** — the rule is an interaction (`decent_position AND ctr_underperforming`, both gating a demand weight). A model that can't learn interactions has no business claiming to beat a rule built entirely out of them.
- **Decision Tree (max_depth=4)** — trained to sanity-check the forest, not to win: if a 4-node tree gets close to the forest's score, the forest's extra complexity isn't earning its keep, and the assignment explicitly says not to reward complexity alone.

Final single choice gets made in Section 3 after seeing the actual comparison table.

In [2]:
# --- Build FEATURE_MONTH (March) aggregate -- fact-table only, mirrors w04's scored table exactly ---
feature_month_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           AVG(gsc_avg_position) AS avg_position_month
    FROM read_parquet('{DAILY_FEATURE}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
feature_month_df['ctr_month'] = feature_month_df['clicks_month'] / feature_month_df['impressions_month'] * 100

def position_bucket(pos):
    if pos <= 0: return '0_no_position_data'
    elif pos <= 3: return '1_top3'
    elif pos <= 10: return '2_page1_4_10'
    elif pos <= 20: return '3_page2_11_20'
    elif pos <= 50: return '4_page3plus_21_50'
    else: return '5_deep_50plus'

feature_month_df['position_bucket'] = feature_month_df['avg_position_month'].apply(position_bucket)

# --- Build LABEL_MONTH (April) aggregate -- impressions only, that's all the label needs ---
label_month_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_next
    FROM read_parquet('{DAILY_LABEL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

before_join_n = len(feature_month_df)
data = feature_month_df.merge(label_month_df, on=['content_hash_id', 'client_hash_id'], how='inner')
after_join_n = len(data)
dropped_n = before_join_n - after_join_n
print(f'March-eligible rows: {before_join_n}  (compare: w04 scored {176738} rows on the same month -- should be close)')
print(f'Rows with April data too (inner join): {after_join_n}  ({dropped_n} dropped, {dropped_n/before_join_n*100:.1f}% -- '
      f'the churn-bias gap: pages that vanish between months are invisible to this label, likely the worst-outcome pages)')

data['pct_change'] = (data['impressions_next'] - data['impressions_month']) / data['impressions_month'] * 100
data['is_declining_next'] = (data['pct_change'] < -20).astype(int)

print()
print('Class balance:')
print(data['is_declining_next'].value_counts(normalize=True).rename('share'))
print()
print('Distinct clients:', data['client_hash_id'].nunique())
print('Rows per client (median):', data.groupby('client_hash_id').size().median())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March-eligible rows: 176738  (compare: w04 scored 176738 rows on the same month -- should be close)
Rows with April data too (inner join): 158549  (18189 dropped, 10.3% -- the churn-bias gap: pages that vanish between months are invisible to this label, likely the worst-outcome pages)

Class balance:
is_declining_next
0    0.521845
1    0.478155
Name: share, dtype: float64

Distinct clients: 46
Rows per client (median): 1007.5


*(Fill in after running -- one sentence: how does the class balance compare to the starter kit's 54.2% declining? If it's badly imbalanced (e.g. <15% or >85%), say so and lean on precision/recall/PR-AUC over accuracy in Section 3 -- accuracy on an imbalanced label is a vanity number.)*

## 2. Split design

**Grouped by client, not random.** `client_hash_id` is for grouping only, per the data contract -- a random row split would let the same client's pages sit in both train and test, so the model could learn *that client's* baseline noise instead of a generalizable pattern. `GroupShuffleSplit` keeps every client entirely on one side.

**Not time-aware on top of that, and here's why that's still honest:** the label is already time-separated (March features, April outcome) by construction -- the leakage risk this split guards against is client-identity leakage, which is orthogonal to the March→April boundary already baked into every row.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(data, groups=data['client_hash_id']))

train_df = data.iloc[train_idx].reset_index(drop=True)
test_df  = data.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
assert len(overlap) == 0, f'Client leakage across the split: {overlap}'

print(f'Train: {len(train_df)} rows, {train_df["client_hash_id"].nunique()} clients')
print(f'Test:  {len(test_df)} rows, {test_df["client_hash_id"].nunique()} clients')
print(f'Test label rate: {test_df["is_declining_next"].mean()*100:.1f}%  (compare to train: {train_df["is_declining_next"].mean()*100:.1f}% -- '
      f'a big gap here means this client split accidentally produced an unrepresentative test set)')

Train: 135314 rows, 34 clients
Test:  23235 rows, 12 clients
Test label rate: 47.0%  (compare to train: 48.0% -- a big gap here means this client split accidentally produced an unrepresentative test set)


## 3. Train + compare vs my baseline

**The baseline is re-scored on `test_df`, using its own committed logic -- but its one data-dependent piece (`expected_ctr_by_bucket`, the per-position-band average CTR the rule compares against) is recalibrated from `train_df` only, not the full dataset.** The w04 notebook calibrated it on the entire 176,738-row slice, which is a small form of test-set peeking once a held-out split exists -- the model is being fit on train only, so the rule's one fitted piece has to follow the same rule or the comparison isn't fair. The fixed thresholds (`position <= 20`, `impressions >= 300`, `0.7x` band average) are constants, not refit.

**Metric: precision@K (K=10, 50, 200) plus precision/recall/F1/ROC-AUC.** Precision@K matters most because this lane is a priority queue someone acts on top-down, not a general classifier.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# --- 1. Re-score the ML-07 rule on test_df, band averages calibrated on train_df only ---
band_avg_ctr_train = train_df.groupby('position_bucket')['ctr_month'].mean()
overall_train_avg = train_df['ctr_month'].mean()  # fallback for any bucket unseen in train

t = test_df.copy()
t['expected_ctr'] = t['position_bucket'].map(band_avg_ctr_train).fillna(overall_train_avg)
decent_position = (t['avg_position_month'] > 0) & (t['avg_position_month'] <= 20)
ctr_underperforming = t['ctr_month'] < (t['expected_ctr'] * 0.7)
t['baseline_score'] = decent_position.astype(int) * ctr_underperforming.astype(int) * t['impressions_month']
t['baseline_action'] = (t['baseline_score'] > 0).astype(int)  # 1 = review_for_refresh

print('Baseline action counts on test set:')
print(t['baseline_action'].value_counts())
print('(if this is all one value, the recalibrated rule degenerated on this split -- stop and check band_avg_ctr_train)')

# --- 2. Features (March-only fact-table columns, leak-free by construction; dim_content excluded, see intro) ---
num_features = ['impressions_month', 'clicks_month', 'avg_position_month', 'ctr_month']
cat_features = ['position_bucket']

for df_ in (train_df, test_df):
    df_['log_impressions_month'] = np.log1p(df_['impressions_month'])
    df_['log_clicks_month'] = np.log1p(df_['clicks_month'])
num_features = num_features + ['log_impressions_month', 'log_clicks_month']

X_train, y_train = train_df[num_features + cat_features], train_df['is_declining_next']
X_test,  y_test  = test_df[num_features + cat_features],  test_df['is_declining_next']

pre = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
])

models = {
    'logistic_regression': Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
    'decision_tree_shallow': Pipeline([('pre', pre), ('clf', DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42))]),
    'random_forest': Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1))]),
}

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

rows = []
y_test_arr = y_test.values

rows.append({
    'method': 'ML-07 rule (recalibrated on train, scored on test)',
    'precision': precision_score(y_test_arr, t['baseline_action'], zero_division=0),
    'recall': recall_score(y_test_arr, t['baseline_action'], zero_division=0),
    'f1': f1_score(y_test_arr, t['baseline_action'], zero_division=0),
    'roc_auc': roc_auc_score(y_test_arr, t['baseline_score']),
    'p_at_10': precision_at_k(y_test_arr, t['baseline_score'], 10),
    'p_at_50': precision_at_k(y_test_arr, t['baseline_score'], 50),
    'p_at_200': precision_at_k(y_test_arr, t['baseline_score'], 200),
})

fitted = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    rows.append({
        'method': name,
        'precision': precision_score(y_test_arr, pred, zero_division=0),
        'recall': recall_score(y_test_arr, pred, zero_division=0),
        'f1': f1_score(y_test_arr, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test_arr, proba),
        'p_at_10': precision_at_k(y_test_arr, proba, 10),
        'p_at_50': precision_at_k(y_test_arr, proba, 50),
        'p_at_200': precision_at_k(y_test_arr, proba, 200),
    })

comparison = pd.DataFrame(rows).set_index('method').round(3)
comparison

Baseline action counts on test set:
baseline_action
0    12219
1    11016
Name: count, dtype: int64
(if this is all one value, the recalibrated rule degenerated on this split -- stop and check band_avg_ctr_train)


,precision,recall,f1,roc_auc,p_at_10,p_at_50,p_at_200
method,,,,,,,
"ML-07 rule (recalibrated on train, scored on test)",0.526,0.531,0.529,0.583,0.7,0.54,0.525
logistic_regression,0.586,0.647,0.615,0.663,0.9,0.74,0.695
decision_tree_shallow,0.565,0.731,0.637,0.626,0.2,0.16,0.330
random_forest,0.558,0.690,0.617,0.638,0.7,0.74,0.720


*(Fill in after running -- name the actual winner and margin, e.g. "random_forest beats the ML-07 rule on p_at_50 (X vs Y) and F1 (X vs Y), but the shallow decision tree is within Z of the forest on p_at_50 -- meaning most of the forest's edge comes from [interaction/feature], not raw complexity." If the rule wins or ties on p_at_10/50, say that plainly -- a hand-rule beating a trained model is a real, reportable result, not a bug to bury. Also state explicitly: did recalibrating the band averages on train-only change the rule's test-set behavior much from what w04's full-dataset calibration would have shown? A big difference there is itself a finding about how sensitive the rule's threshold is to which rows it's calibrated on.)*

## 4. Errors and interpretation

*Permutation importance for the chosen model, plus where the rule and the model disagree.*

In [5]:
from sklearn.inspection import permutation_importance

# Set this to whichever method Section 3's table actually justified -- don't default to random_forest out of habit.
CHOSEN_MODEL = 'random_forest'
chosen = fitted[CHOSEN_MODEL]

perm = permutation_importance(chosen, X_test, y_test, n_repeats=20, random_state=42, scoring='roc_auc', n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print('Permutation importance (ROC-AUC drop):')
print(importance.round(4))

t['model_pred'] = chosen.predict(X_test)
t['true_label'] = y_test_arr

model_right_rule_wrong = t[(t['model_pred'] == t['true_label']) & (t['baseline_action'] != t['true_label'])]
rule_right_model_wrong = t[(t['baseline_action'] == t['true_label']) & (t['model_pred'] != t['true_label'])]

print()
print(f'Model correct, rule wrong: {len(model_right_rule_wrong)} rows')
print(model_right_rule_wrong[['impressions_month','avg_position_month','ctr_month','position_bucket']].describe(include='all'))
print()
print(f'Rule correct, model wrong: {len(rule_right_model_wrong)} rows')
print(rule_right_model_wrong[['impressions_month','avg_position_month','ctr_month','position_bucket']].describe(include='all'))

Permutation importance (ROC-AUC drop):
ctr_month                0.0522
log_impressions_month    0.0256
impressions_month        0.0246
log_clicks_month         0.0045
clicks_month             0.0032
avg_position_month      -0.0057
position_bucket         -0.0068
dtype: float64

Model correct, rule wrong: 4019 rows
        impressions_month  avg_position_month    ctr_month    position_bucket
count         4019.000000         4019.000000  4019.000000               4019
unique                NaN                 NaN          NaN                  5
top                   NaN                 NaN          NaN  4_page3plus_21_50
freq                  NaN                 NaN          NaN               1579
mean          1130.531724           18.800145     0.167327                NaN
std           3530.038008           14.950764     0.567563                NaN
min              1.000000            0.200000     0.000000                NaN
25%             14.000000            6.558798     0.000000  

*(Fill in after running -- two or three sentences, concrete: what does `importance` say the model leans on -- is it `ctr_month` doing most of the work (same signal the rule uses), or something the rule structurally ignores like raw `impressions_month` scale or `log_clicks_month`? For the disagreement rows: name the actual pattern (e.g. "the model catches high-impression, moderate-CTR-gap pages just under the rule's 0.7x threshold that the rule's hard cutoff misses entirely") -- don't just report the row counts.)*

**Also required, not optional:** revisit the churn-bias gap from Section 1 -- state the real dropped-row count and one sentence on which direction it likely biases the comparison above (does excluding vanished pages make the model or the rule look artificially better, by removing the worst-outcome pages from both?). And note the extreme-impression/near-zero-CTR pattern flagged in the w04 top-10 review (row 3, 212k impressions at 0.011% CTR) -- check whether that pattern shows up in either error group here; if the rule is still weighting those pages heavily via raw `impressions_month`, that's the same weak-pick risk carrying forward into this comparison.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it (no `[fill in]` placeholders left)
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) — confirm `execution_count` is populated and outputs are present
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The comparison table in Section 3 uses the same test rows, same label, and same metric for the rule and every model — not different subsets
- [x] The baseline's band-average recalibration used train_df only, not the full dataset
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.